# Start here: what's running and how notebooks reach Spark

Nothing in this Jupyter container runs Spark. Every Spark job goes the same way:

```
notebook  --REST :10099-->  Kyuubi  --spark-submit-->  Spark master/worker
notebook  --JDBC :10009-->  Kyuubi  --SQL engine---->  Spark (one engine per user)
```

The notebooks use `client/kyuubi_lab.py`, the same code as `make test` and the Airflow
DAGs, so a job behaves the same wherever you start it from.

| notebook | what it covers |
|---|---|
| 01 | Kyuubi REST by hand: JSON batch, multipart upload, logs, kill, failure |
| 02 | SQL over Kyuubi JDBC: databases, CTAS in Parquet/ORC, EXPLAIN, session conf |
| 03 | file formats (CSV, Parquet, ORC, JSON, Avro) and Hive ELT with RCFILE |
| 04 | Postgres: file -> Postgres, Postgres -> files, Postgres -> Postgres |
| 05 | Iceberg, Delta and Hudi |
| 06 | Kafka producer -> Spark consumer |
| 07 | Spark publishes to Kafka (enrich, windowed metrics, batch backfill) |
| 08 | Structured Streaming on file events |
| 09 | the real-time order tracking project |
| 10 | always-on streams: start, feed, watch, stop |
| 11 | trigger Airflow DAGs from a notebook (Airflow -> Kyuubi -> Spark) |
| 12 | everything, end to end, with a pass/fail table |

In [ ]:
import kyuubi_lab as lab          # client/kyuubi_lab.py (on PYTHONPATH in the Jupyter image)
import pandas as pd
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 200)
RID = lab.new_run_id()            # one id for every job in this notebook run
print("Kyuubi REST", lab.REST_URL, "| JDBC", f"{lab.JDBC_HOST}:{lab.JDBC_PORT}", "| user", lab.USER, "| run", RID)

## Which services can this notebook reach?
`kafka` and `airflow` are optional (`make kafka-install`, `make airflow`).

In [ ]:
lab.services()

## The job catalog
Every example job in the lab. `run(name)` runs one of them.

In [ ]:
lab.jobs()

## What a job sends to Kyuubi
`payload()` builds the REST body without sending it. This is the same JSON as in `All_execution_payload.md`.

In [ ]:
import json
print(json.dumps(lab.payload('iceberg', RID)['body'], indent=2))

## A first job
The smoke test: PySpark file from RustFS, a few tasks on the executors.

In [ ]:
r = lab.run('py-smoke', run_id=RID)
lab.log_tail(r, 'SMOKE')

## Web UIs (from your Mac)
* Kyuubi: http://localhost:10099/ui
* Spark master: http://localhost:8080, history: http://localhost:18080
* RustFS console: http://localhost:9101
* Kafka UI: http://localhost:8090 (if installed)
* Airflow: http://localhost:8085 (if running)